# Minimum and Maximum Temperature

Historical analysis of annual minimum (`TMIN`) and maximum (`TMAX`) surface air temperature at the selected GHCN station, including the diurnal temperature range (`diff = TMAX − TMIN`).

**Indicator:** annual mean of daily TMIN and TMAX, with linear trends; diurnal range as a measure of day–night temperature contrast.

```{glue:figure} trend_fig_max_min
:scale: 50%
:align: right
```

**Figure. Annual maximum (red) and minimum (blue) temperature.** A solid black line indicates a statistically significant trend (p < 0.05); a dashed line indicates a non-significant trend.


## Setup

Import libraries and helper functions. Shared plotting utilities come from the [indicators_setup](https://github.com/lauracagigal/indicators_setup) repository; site configuration and output helpers come from `functions/air_temp.py`.


In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os.path as op
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import plotly.io as pio
pio.renderers.default = "notebook_connected"

from myst_nb import glue

sys.path.append("../../../../../indicators_setup")
from ind_setup.plotting_int import plot_timeseries_interactive, fig_int_to_glue

sys.path.append("../../../functions")
from data_downloaders import GHCN
from air_temp import (
    load_site_config,
    site_config_filename,
    list_available_sites,
    build_output_filename,
    build_site_figures_dir,
    persist_minmax_temperature_outputs,
)

### Define location and variables of interest

Set `site_key` to one of the sites already configured with [`../00_site_setup.ipynb`](../00_site_setup.ipynb) (see the list printed below). The notebook loads coordinates, station ID, reference period and paths from `data/sites/<site_key>.json` — the filename is resolved with `site_config_filename()`, the same helper `00_site_setup.ipynb` uses to save it, so any site name (accents, spaces, capitals) resolves consistently.


In [3]:
sites_dir = Path('../../../data/sites')
available_sites = list_available_sites(sites_dir)
print(f"{len(available_sites)} site(s) already configured in {sites_dir}:")
display(available_sites)

1 site(s) already configured in ../../../data/sites:


,site_key,site_name,country,ghcn_station_id,ghcn_station_name,vars_interest
0,palau_psw00040309,palau_PSW00040309,Palau,PSW00040309,PW KOROR GSN 91408,"[TMIN, TMAX, PRCP]"


In [4]:
site_key = "palau_psw00040309"  # pick a site_key from the table printed above

site_config_path = sites_dir / site_config_filename(site_key)
site_cfg = load_site_config(site_config_path)

site_name = site_cfg.get('site_name', 'Site')
site_lon = float(site_cfg['site_lon'])
site_lat = float(site_cfg['site_lat'])
country = site_cfg['country']
ghcn_station_id = site_cfg['ghcn_station_id']
ghcn_station_name = site_cfg.get('ghcn_station_name', '')
vars_interest = list(site_cfg.get('vars_interest', ['TMIN', 'TMAX']))
station_label = ghcn_station_name or site_name
glue("station_label", station_label, display=False)

### Get Data

Load the cached daily temperature series from `data/air_temp/GHCN_<station_id>.pkl`. All download and quality-control steps were completed in [`../00_site_setup.ipynb`](../00_site_setup.ipynb).


In [5]:
data_base_dir = Path('../../../data')
site_figures_dir = build_site_figures_dir(Path('../../../outputs'), site_name, site_lon, site_lat)
data_dir = Path(data_base_dir, 'air_temp')
output_dir = data_base_dir

output_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

path_data = str(data_dir)

### Observations from Station

Daily minimum and maximum temperature records for the GHCN station defined in the site configuration.


[GHCN-Daily documentation](https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/doc/GHCND_documentation.pdf)


The data used for this analysis comes from the [GHCN-Daily](https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/) database (NOAA NCEI).

GHCN-Daily provides historical daily climate records over global land areas. Records from numerous sources are merged and subjected to quality-assurance reviews. The variables used here are **`TMIN`** and **`TMAX`** (daily minimum and maximum temperature, °C), with **`diff = TMAX − TMIN`** as the diurnal range.

**Data loading:** this notebook reads the pre-processed pickle created by [`../00_site_setup.ipynb`](../00_site_setup.ipynb). It does **not** re-download from NOAA.

**Outputs:** figures → `outputs/figures/<site_tag>/`; tables and JSON → `outputs/tables/<site_tag>/`.


In [6]:
print(f"Site: {site_name} ({country}) - GHCN station {ghcn_station_id} {ghcn_station_name}")

Site: palau_PSW00040309 (Palau) - GHCN station PSW00040309 PW KOROR GSN 91408


In [7]:
pickle_path = op.join(path_data, f'GHCN_{ghcn_station_id}.pkl')
st_data = pd.read_pickle(pickle_path)
print(f'Loaded {len(st_data)} daily records from {pickle_path}')

Loaded 25966 daily records from ../../../data/air_temp/GHCN_PSW00040309.pkl


In [8]:
st_data_daily = st_data.copy()

In [9]:
print(st_data_daily.TMIN.mean(), st_data_daily.TMAX.mean())

24.39285989370715 31.05149041053685


In [10]:
dict_plot = [{'data' : st_data_daily, 'var' : 'TMAX', 'ax' : 1, 'label' : 'TMAX'}]
fig = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12))


In [11]:
dict_plot = [{'data' : st_data_daily, 'var' : 'TMIN', 'ax' : 1, 'label' : 'TMIN'}]
fig = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12))


In [12]:
st_data = st_data.resample('YE').mean()
glue("n_years", len(np.unique(st_data.index.year)), display=False)
glue("start_year", st_data.dropna().index[0].year, display=False)
glue("end_year", st_data.dropna().index[-1].year, display=False)

## Analysis

All indicators below are computed from the loaded daily `TMIN` / `TMAX` series after annual aggregation. Linear trends are estimated with `plot_timeseries_interactive` from `indicators_setup`.


### Annual minimum and maximum temperature


Time series of annual mean daily minimum and maximum temperature with fitted trends. Figures are saved under `outputs/figures/<site_tag>/` (e.g. `F3_ST_min_<site_tag>.html`, `F3_ST_max_<site_tag>.html`, `F3_ST_min_max_<site_tag>.html`).


#### Minimum temperature (`TMIN`)

Annual mean of daily minimum temperature with a fitted linear trend.


In [13]:
dict_plot = [{'data' : st_data, 'var' : 'TMIN', 'ax' : 1, 'label' : 'TMIN'},
        ]

In [14]:
fig, trend_minimum = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (24, 11), return_trend = True)
fig.write_html(
    site_figures_dir / build_output_filename('F3_ST_min', site_name, site_lon, site_lat, ext='html'),
    include_plotlyjs="cdn",
)

#### Maximum temperature (`TMAX`)

Annual mean of daily maximum temperature with a fitted linear trend.


In [15]:
dict_plot = [{'data' : st_data, 'var' : 'TMAX', 'ax' : 1, 'label' : 'TMAX'},
        ]

In [16]:
fig, trend_maximum = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (24, 11), return_trend = True)
fig.write_html(
    site_figures_dir / build_output_filename('F3_ST_max', site_name, site_lon, site_lat, ext='html'),
    include_plotlyjs="cdn",
)

In [17]:
print(st_data.TMIN.mean(), st_data.TMAX.mean())

24.4096970843879 31.033677983557787


#### Minimum and maximum together

Combined plot of annual `TMIN` (blue) and `TMAX` (red) for comparison of trends and spread.


In [18]:
dict_plot = [{'data' : st_data, 'var' : 'TMIN', 'ax' : 1, 'label' : 'TMIN'},
        {'data' : st_data, 'var' : 'TMAX', 'ax' : 1, 'label' : 'TMAX'},
        # {'data' : st_data, 'var' : 'diff', 'ax' : 1, 'label' : 'Difference TMAX - TMIN'}
        ]

In [19]:
fig, TRENDS = plot_timeseries_interactive(dict_plot, label_yaxes = 'Temperature [ºC]', trendline=True, figsize = (24, 11), return_trend = True)
fig.write_html(
    site_figures_dir / build_output_filename('F3_ST_min_max', site_name, site_lon, site_lat, ext='html'),
    include_plotlyjs="cdn",
)
fig.write_image(
    site_figures_dir / build_output_filename('F3_ST_min_max', site_name, site_lon, site_lat),
    width=1200, height=600,
)

glue("trend_min", float(TRENDS[0]), display=False)
glue("trend_max", float(TRENDS[1]), display=False)

glue("change_min", float(TRENDS[0]*len(np.unique(st_data.index.year))), display=False)
glue("change_max", float(TRENDS[1]*len(np.unique(st_data.index.year))), display=False)

glue("trend_fig_max_min", fig_int_to_glue(fig), display=False)

```{glue:figure} trend_fig_max_min
:name: "F3_ST_min_max"

Annual maximum (red) and minimum (blue) temperature. A solid black line indicates a statistically significant trend (p < 0.05); a dashed line indicates a non-significant trend.
```


### Diurnal temperature range


Annual mean of `diff = TMAX − TMIN`. A decreasing trend indicates a narrowing day–night temperature contrast over time. Saved as part of the F3 figure set under `outputs/figures/<site_tag>/`.


In [20]:
dict_plot = [{'data' : st_data, 'var' : 'diff', 'ax' : 1, 'label' : 'Difference TMAX - TMIN'}]
fig, trend = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12), return_trend = True)
glue("trend_diff", float(trend[0]), display=False)


Annual mean diurnal temperature range (`TMAX − TMIN`) with a fitted trend.


### Summary table and persisted outputs

The final cell builds the summary table and calls `persist_minmax_temperature_outputs()` to save:

- `T_minmax_annual_<site_tag>.csv`
- `T_minmax_summary_table_<site_tag>.csv`
- `T_minmax_summary_metrics_<site_tag>.json`


In [21]:
from ind_setup.tables import table_temp_12

summary_table = table_temp_12(st_data, st_data_daily, trend_maximum[0], trend_minimum[0])
persist_minmax_temperature_outputs(
    Path('../../../outputs'),
    site_name, site_lon, site_lat,
    ghcn_station_id, ghcn_station_name, country,
    st_data, summary_table,
    trend_minimum, trend_maximum, trend,
)

Metric,Value
Annual Maximum Temperature (°C),32.104
Change in Annual Maximum Temperature since 1951,0.150
Rate of Change in Annual Maximum Temperature (°C/year),0.002
Annual Minimum Temperature (°C),23.757
Change in Annual Minimum Temperature since 1951,1.125
Rate of Change in Annual Minimum Temperature (°C/year),0.015
Mean Daily Mean Temperature (°C),27.722
Mean Daily Maximum Temperature (°C),31.051
Mean Daily Minimum Temperature (°C),24.393


PosixPath('../../../outputs/tables/palau_psw00040309_lat7p337_lon134p477')